# plotlet in a Jupyter notebook

plotlet is a small Python library that emits matplotlib-style SVG plots. Artist methods (`line`, `scatter`, `title`, ...) only *record* what to draw; the SVG is emitted when the chart auto-renders as a cell's last expression or you call `.show()`.

In [1]:
import plotlet as pt
import math

xs = [i * 0.1 for i in range(64)]
df = {
    "time":  xs,
    "sin_x": [math.sin(t) for t in xs],
    "cos_x": [math.cos(t) for t in xs],
}

c = pt.chart(df, title="Hello from plotlet", xlabel="x", ylabel="y",
             legend=True, grid=True)
c.line(x="time", y="sin_x", label="sin(x)")
c.line(x="time", y="cos_x", label="cos(x)", linestyle="--")

Notice we didn't call `.show()` — when the last expression in a cell is a `Figure`, Jupyter's `_repr_html_` protocol auto-renders it. Same as plotly / pandas DataFrames.

`.show()` works too if you want to render mid-cell:

In [2]:
df = {"cat": ["A", "B", "C", "D"], "count": [4, 7, 2, 9]}

c = pt.chart(df, width=500, height=300, title="explicit show()")
c.bar(x="cat", y="count", color="C2")
c.show()

## With numpy / pandas
Any object with `.tolist()` is auto-converted, so numpy arrays and pandas Series Just Work.

In [3]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(0)

xs = np.linspace(0, 6, 60)
df = pd.DataFrame({
    "x":      np.concatenate([xs, xs]),
    "y":      np.concatenate([np.sin(xs), np.cos(xs)]) + rng.normal(0, 0.15, 120),
    "series": ["sin"] * len(xs) + ["cos"] * len(xs),
})

c = pt.chart(df, xlabel="x", ylabel="y", legend=True, grid=True)
c.scatter(x="x", y="y", hue="series", s=25, alpha=0.6)

## Histogram

In [4]:
import random
random.seed(0)

df = {"value": [random.gauss(0, 1) for _ in range(2000)]}

c = pt.chart(df, title="normal(0, 1)")
c.hist(x="value", bins=30, color="C3")

## Confidence bands (`fill_between`)

In [5]:
xs    = [i * 0.1 for i in range(64)]
mean  = [math.sin(t) for t in xs]
lower = [m - 0.25 for m in mean]
upper = [m + 0.25 for m in mean]

c = pt.chart({"x": xs, "mean": mean, "lo": lower, "hi": upper},
             xlabel="x", ylabel="y", legend=True)
c.fill_between(x="x", y1="lo", y2="hi", color="C0", alpha=0.2, label="±0.25")
c.line(x="x", y="mean", label="sin(x)")

## Reference lines (`axhline`, `axvline`)
Decorate the frame without affecting autoscale.

In [6]:
xs = [i * 0.1 for i in range(64)]
df = {"x": xs, "y": [math.sin(t) for t in xs]}

c = pt.chart(df, xlabel="x", ylabel="y", legend=True)
c.line(x="x", y="y", label="sin(x)")
c.axhline(0, linestyle="--", color="gray")
c.axvline(math.pi, linestyle=":", color="C3", label="x = π")

## Shaded regions (`axhspan`, `axvspan`)

In [7]:
xs = [i * 0.1 for i in range(64)]
df = {"x": xs, "y": [math.sin(t) for t in xs]}

c = pt.chart(df, xlabel="x", ylabel="y")
c.axhspan(-0.5, 0.5, color="C0", alpha=0.1)
c.axvspan(math.pi, 2 * math.pi, color="C2", alpha=0.1)
c.line(x="x", y="y")

## Heatmaps (`imshow`)
Pass any 2-D array; `cmap=` selects from the 180 vendored matplotlib colormaps.

In [8]:
y, x = np.mgrid[-3:3:60j, -3:3:60j]
z = np.exp(-(x**2 + y**2) / 2) * np.cos(3 * x) * np.cos(3 * y)

c = pt.chart(title="2-D field", xlabel="x", ylabel="y")
c.imshow(z, extent=[-3, 3, -3, 3], cmap="viridis")